### [Video de apoyo a esta lectura experimental]()

Para demostrarles a tus estudiantes de una forma muy visual e impactante la diferencia entre procesar datos con **Pandas (en memoria RAM)** frente a **SQL (basado en páginas de disco)**, podemos diseñar un experimento práctico.

El objetivo es simular un escenario donde Pandas sufriría por el uso de memoria, mientras que **SQLite** resolverá la consulta de forma inmediata utilizando índices y paginación.

A continuación, te presento el código completo estructurado para un cuaderno Jupyter. Este script genera los tres archivos `.csv` con datos ficticios adaptados a **Caucasia**, los consolida en un archivo `.db`, y finalmente ejecuta un benchmark (prueba de rendimiento) comparando ambos enfoques.

---

## 1. Generación de los archivos .csv de prueba

Primero, creamos un script que genere datos sintéticos. Para que la diferencia de rendimiento sea notoria sin necesidad de descargar gigabytes de información, crearemos estructuras que simulen un volumen considerable de transacciones históricas.

```python
import pandas as pd
import numpy as np
import sqlite3
import time
import os

# Configuración para reproducibilidad
np.random.seed(42)
n_clientes = 10000       # 10 mil clientes maestros
n_operaciones = 1000000  # 1 millón de registros de transacciones e historial

print("Generando archivos CSV de prueba...")

# ARCHIVO 1: clientes.csv (Datos maestros estáticos)
sectores = ['Ganadería/Agro', 'Minería/Servicios', 'Comercio Minorista', 'Independientes']
df_clientes = pd.DataFrame({
    'identificador_cliente': np.arange(1001, 1001 + n_clientes),
    'sector_economico': np.random.choice(sectores, n_clientes),
    'fecha_afiliacion': pd.date_range(start='2020-01-01', periods=n_clientes, freq='h').strftime('%Y-%m-%d')
})
df_clientes.to_csv('clientes.csv', index=False)

# ARCHIVO 2: ingresos_credito.csv (Datos financieros)
df_ingresos = pd.DataFrame({
    'identificador_cliente': np.arange(1001, 1001 + n_clientes),
    'ingresos_mensuales_cop': np.random.uniform(1300000, 8500000, n_clientes).round(2),
    'score_credito': np.random.randint(300, 850, n_clientes)
})
df_ingresos.to_csv('ingresos_credito.csv', index=False)

# ARCHIVO 3: satisfaccion_operaciones.csv (Historial masivo de interacciones)
# Nota: Generamos 1 millón de filas para saturar la búsqueda secuencial de Pandas
df_satisfaccion = pd.DataFrame({
    'id_operacion': np.arange(1, n_operaciones + 1),
    'identificador_cliente': np.random.randint(1001, 1001 + n_clientes, n_operaciones),
    'satisfaccion': np.random.choice([1.0, 2.0, 3.0, 4.0, 5.0, np.nan], n_operaciones, p=[0.1, 0.1, 0.2, 0.3, 0.2, 0.1])
})
df_satisfaccion.to_csv('satisfaccion_operaciones.csv', index=False)

print("¡Archivos CSV creados exitosamente!")

```

---

## 2. Consolidación en el archivo .db con Indexación

Aquí es donde ocurre la magia que les debes explicar: creamos el contenedor `.db` e **inyectamos índices estratégicos**. El índice es el secreto detrás de la velocidad de SQL.

```python
# Conectar a SQLite (crea el archivo si no existe)
conexion = sqlite3.connect('caucasia_negocios.db')
cursor = conexion.cursor()

print("Consolidando archivos en caucasia_negocios.db...")

# Cargar y enviar a SQL
pd.read_csv('clientes.csv').to_sql('clientes', conexion, if_exists='replace', index=False)
pd.read_csv('ingresos_credito.csv').to_sql('ingresos_credito', conexion, if_exists='replace', index=False)
pd.read_csv('satisfaccion_operaciones.csv').to_sql('satisfaccion_operaciones', conexion, if_exists='replace', index=False)

# CRUCIAL PARA LA EFICIENCIA: Creación de índices en las llaves de búsqueda
cursor.execute("CREATE INDEX IF NOT EXISTS idx_clientes_id ON clientes(identificador_cliente);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_ingresos_id ON ingresos_credito(identificador_cliente);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_satisfaccion_id ON satisfaccion_operaciones(identificador_cliente);")

conexion.commit()
print("¡Base de datos consolidada e indexada exitosamente!")

```

---

## 3. El Experimento en Clase: El Benchmark

Pídeles a tus estudiantes que busquen un cliente específico (por ejemplo, el `identificador_cliente = 5555`) haciendo un cruce de información (`JOIN` / `merge`) para ver sus ingresos, sector y su historial de encuestas. Mide el tiempo de ambos métodos.

```python
cliente_a_buscar = 5555

# =====================================================================
# ENFOQUE A: PANDAS (Carga todo en RAM y busca secuencialmente)
# =====================================================================
start_pandas = time.time()

# Pandas se ve obligado a leer los archivos completos del disco a la RAM
p_clientes = pd.read_csv('clientes.csv')
p_ingresos = pd.read_csv('ingresos_credito.csv')
p_satisfaccion = pd.read_csv('satisfaccion_operaciones.csv')

# Realiza el cruce en memoria de las tablas completas
df_fusionado = p_satisfaccion.merge(p_clientes, on='identificador_cliente').merge(p_ingresos, on='identificador_cliente')

# Filtra la fila del cliente
resultado_pandas = df_fusionado[df_fusionado['identificador_cliente'] == cliente_a_buscar]

end_pandas = time.time()
tiempo_pandas = end_pandas - start_pandas


# =====================================================================
# ENFOQUE B: SQL (Paginación e Índices B-Tree)
# =====================================================================
start_sql = time.time()

# El motor va directo a las páginas del disco correspondientes gracias al índice
consulta_sql = """
SELECT s.*, c.sector_economico, i.ingresos_mensuales_cop 
FROM satisfaccion_operaciones s
JOIN clientes c ON s.identificador_cliente = c.identificador_cliente
JOIN ingresos_credito i ON s.identificador_cliente = i.identificador_cliente
WHERE s.identificador_cliente = ?;
"""
resultado_sql = pd.read_sql_query(consulta_sql, conexion, params=(cliente_a_buscar,))

end_sql = time.time()
tiempo_sql = end_sql - start_sql

# Cerrar conexión al terminar
conexion.close()

# =====================================================================
# RESULTADOS
# =====================================================================
print("\n" + "="*50)
print("             TABLA DE RENDIMIENTO")
print("="*50)
print(f"Tiempo con Pandas (RAM): {tiempo_pandas:.4f} segundos")
print(f"Tiempo con SQL (Disco):  {tiempo_sql:.4f} segundos")
print("-"*50)
print(f"¡SQL fue {tiempo_pandas / tiempo_sql:.1f} veces más rápido!")
print("="*50)

```

---

## 💡 Guión pedagógico para explicar el resultado a los alumnos:

Cuando ejecuten la celda del benchmark, verán que **SQL resuelve la consulta en milisegundos**, siendo drásticamente más rápido que Pandas. Explícales los dos motivos clave:

1. **La ilusión de la memoria RAM:** Explícales que para obtener los datos de *un solo cliente*, Pandas tuvo que parsear y estructurar en la memoria RAM el millón de filas de transacciones completas. Si el archivo pesara 20 GB en lugar de unos cuantos megabytes, la computadora de la clase se colgaría (`MemoryError`).
2. **El Vigilante Indexado:** SQL no leyó el millón de filas. Gracias a la instrucción `CREATE INDEX`, el motor tiene un mapa ordenado (Árbol Binario o *B-Tree*). Saltó directamente al bloque físico del disco duro donde residía el cliente `5555`, cargó únicamente esas cuantas filas a la RAM y descartó el resto del archivo inmediatamente.